# 3D Pose Avatar

In this notebook, your webcam and two small neural networks turn you into a puppeteer. By the end, a wooden mannequin in a 3D scene copies every move you make: wave, lean, squat or turn around, and it does the same.

We'll build it in three steps:

1. **See the pose.** Find your body in the webcam picture as a skeleton of 33 points, called landmarks.
2. **Draw the numbers.** Draw those landmarks yourself on a blank canvas, from the front and from the side, to find out what the model knows about where you are in 3D.
3. **Bring it to life.** Use the landmarks to move a mannequin, built from capsules and spheres, in a 3D scene.

Everything runs inside your browser, so your video is never uploaded anywhere. If you haven't used pose detection before, [[089433C4-E561-4C8D-8489-CAB803B7463E|Pose Detection]] explains the models in more detail.

## Setting up pose tracking

The first cell below downloads the two BlazePose models from Hugging Face (about 18 MB), loads them into OpenCV, and defines a function called `detect()`. It runs by itself when the notebook opens. Expand it if you'd like to see how it works.

`detect(frame)` returns a list of poses, one for each person it finds. Each pose is a list of 33 landmarks, and each landmark is a dictionary:

| Key | What it means |
|---|---|
| `x`, `y` | Where the landmark is in the picture, in pixels |
| `visibility` | How sure the model is that this part of the body is in view, from 0 to 1 |
| `world` | Where the landmark is in 3D, as `[x, y, z]` in **metres** from the middle of your hips |

`world` is what makes a 3D avatar possible. We'll explore it in step 2.

The second cell gives names to the landmark numbers we'll use, so that the code can say `LEFT_ELBOW` instead of `13`.

In [ ]:
#@title Set up pose tracking
from huggingface_hub import hf_hub_download
import cv2

person_model_path = hf_hub_download(repo_id="opencv/person_detection_mediapipe", filename="person_detection_mediapipe_2023mar.onnx")
pose_model_path = hf_hub_download(repo_id="opencv/pose_estimation_mediapipe", filename="pose_estimation_mediapipe_2023mar.onnx")
person_net = cv2.dnn.readNet(person_model_path)
pose_net = cv2.dnn.readNet(pose_model_path)

import numpy as np

DETECTOR_SIZE = 224
LANDMARK_SIZE = 256

# The person detector makes 2254 predictions, each relative to a fixed
# "anchor" point: 2 per cell of a 28x28 grid, 2 per cell of 14x14 and 6 per
# cell of 7x7, stored as (x, y) cell centres from 0 to 1.
anchors = []
for grid, per_cell in ((28, 2), (14, 2), (7, 6)):
  xs, ys = np.meshgrid((np.arange(grid) + 0.5) / grid, (np.arange(grid) + 0.5) / grid)
  anchors.append(np.repeat(np.stack((xs.ravel(), ys.ravel()), axis=1), per_cell, axis=0))
anchors = np.concatenate(anchors)

def sigmoid(x):
  """Turn the network's raw scores into probabilities from 0 to 1."""
  return 1 / (1 + np.exp(-np.clip(x, -80, 80)))

def find_people(frame, min_confidence=0.5, overlap_threshold=0.3):
  """Run the person detector. Returns a list of (hip_centre, body_point) pairs:
  the middle of the hips, and a point whose distance from the hips is the
  radius of a circle around the whole body, in the direction of the head."""
  # 1. Letterbox: shrink to fit 224x224 keeping the aspect ratio, padding
  #    both sides with black. The model wants RGB pixels from -1 to 1.
  height, width = frame.shape[:2]
  scale = DETECTOR_SIZE / max(height, width)
  resized = cv2.resize(frame, (int(width * scale), int(height * scale)))
  pad_x = (DETECTOR_SIZE - resized.shape[1]) // 2
  pad_y = (DETECTOR_SIZE - resized.shape[0]) // 2
  padded = cv2.copyMakeBorder(resized, pad_y, DETECTOR_SIZE - resized.shape[0] - pad_y,
                              pad_x, DETECTOR_SIZE - resized.shape[1] - pad_x, cv2.BORDER_CONSTANT)
  blob = cv2.dnn.blobFromImage(padded, 1 / 127.5, mean=(127.5, 127.5, 127.5), swapRB=True)

  # 2. Run the network: 2254 rows of [box cx, cy, w, h, 4 keypoints (x, y)],
  #    offsets in 224-pixel units from each anchor, plus one score per row.
  person_net.setInput(blob)
  output, scores = person_net.forward(['Identity:0', 'Identity_1:0'])
  output = output[0] / DETECTOR_SIZE
  confidences = sigmoid(scores[0, :, 0])

  # 3. Decode into frame pixels: add the anchor, then undo the letterbox.
  def to_frame(xy):
    return (xy + anchors) * DETECTOR_SIZE / scale - (pad_x / scale, pad_y / scale)
  centres = to_frame(output[:, 0:2])
  sizes = output[:, 2:4] * DETECTOR_SIZE / scale
  boxes = np.concatenate([centres - sizes / 2, sizes], axis=1)
  hips = to_frame(output[:, 4:6])
  body_points = to_frame(output[:, 6:8])

  # 4. Non-maximum suppression: drop boxes that overlap a more confident one.
  keep = cv2.dnn.NMSBoxes(boxes.tolist(), confidences.tolist(), min_confidence, overlap_threshold)
  return [(hips[i], body_points[i]) for i in np.array(keep).flatten()]

def upright_crop(frame, centre, up, size, crop_size):
  """Cut a size x size square out of the frame, centred on `centre` and
  rotated so the `up` direction points to the top, then scale it to
  crop_size x crop_size. Returns the crop and the matrix that maps crop
  pixels back to frame pixels."""
  up = up / np.linalg.norm(up)
  right = np.array([-up[1], up[0]])
  half = size / 2
  top_left = centre + up * half - right * half
  top_right = centre + up * half + right * half
  bottom_left = centre - up * half - right * half
  to_crop = cv2.getAffineTransform(np.float32([top_left, top_right, bottom_left]),
                                   np.float32([[0, 0], [crop_size, 0], [0, crop_size]]))
  crop = cv2.warpAffine(frame, to_crop, (crop_size, crop_size))
  return crop, cv2.invertAffineTransform(to_crop)

def find_landmarks(frame, hip, body_point, min_confidence=0.5):
  """Run the landmark model on one person. Returns the 33 landmarks and the
  (hip_centre, body_point) pair for this person in the next frame, or None
  if the model doesn't think there is a person there. Each landmark is
  {"x", "y", "z", "visibility", "world"}: x and y in frame pixels, and
  "world" = [x, y, z] in metres from the middle of the hips."""
  # 1. Crop an upright square around the whole body, a bit bigger than the
  #    detector's circle. The model wants RGB pixels from 0 to 1, in
  #    height x width x channel order (not a blobFromImage blob).
  radius = np.linalg.norm(body_point - hip)
  size = radius * 2 * 1.25
  crop, to_frame = upright_crop(frame, hip, body_point - hip, size, LANDMARK_SIZE)
  blob = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)[np.newaxis].astype(np.float32) / 255

  # 2. Run the network: 39 rows of [x, y, z, visibility, presence] (33 body
  #    landmarks, then 6 the model uses internally), a confidence, and the
  #    same 39 points again as [x, y, z] in metres ("world" landmarks).
  pose_net.setInput(blob)
  landmarks, confidence, world = pose_net.forward(['Identity', 'Identity_1', 'Identity_4'])
  if confidence[0, 0] < min_confidence:
    return None
  landmarks = landmarks[0].reshape(39, 5)
  world = world[0].reshape(39, 3)

  # 3. Map crop pixels back to frame pixels. z is depth relative to the
  #    hips, in the same (crop) pixel units, so scale it the same way.
  xy = np.c_[landmarks[:, :2], np.ones(39)] @ to_frame.T
  z = landmarks[:, 2] * size / LANDMARK_SIZE
  visibility = sigmoid(landmarks[:, 3])
  pose = [{'x': int(x), 'y': int(y), 'z': float(d), 'visibility': round(float(v), 2),
           'world': [round(float(c), 3) for c in w]}
          for (x, y), d, v, w in zip(xy[:33], z, visibility, world)]

  # 4. Two of the hidden points (33 and 34) are the same pair the person
  #    detector gives: the middle of the hips and the body point. So the next
  #    frame can start from them and skip the detector.
  return pose, (xy[33], xy[34])

# The last frame's (hip_centre, body_point) while one person is being followed.
tracked = None

def detect(frame, max_people=1, mirrored=False):
  """Find up to max_people poses in a BGR frame. Each pose is a list of 33
  landmarks, each {"x", "y", "z", "visibility", "world"} (see find_landmarks).
  Pass mirrored=True for a mirrored (selfie) frame: the models are run on the
  un-mirrored frame so that "left" means the person's own left, and x (in
  pixels and in metres) is then flipped to match the picture.
  With max_people=1, a person found in one frame is followed into the next
  without running the person detector again, which is about twice as fast."""
  global tracked
  if mirrored:
    frame = cv2.flip(frame, 1)
  poses = []
  if max_people == 1 and tracked is not None:
    result = find_landmarks(frame, *tracked)
    tracked = None
    if result is not None:
      pose, tracked = result
      poses.append(pose)
  if not poses:
    for hip, body_point in find_people(frame)[:max_people]:
      result = find_landmarks(frame, hip, body_point)
      if result is not None:
        pose, region = result
        poses.append(pose)
        if max_people == 1:
          tracked = region
  if mirrored:
    width = frame.shape[1]
    for pose in poses:
      for landmark in pose:
        landmark['x'] = width - 1 - landmark['x']
        landmark['world'][0] = -landmark['world'][0]
  return poses

print("Pose tracking is ready")

In [ ]:
#@title Landmark names
NOSE = 0
LEFT_EAR, RIGHT_EAR = 7, 8
LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12
LEFT_ELBOW, RIGHT_ELBOW = 13, 14
LEFT_WRIST, RIGHT_WRIST = 15, 16
LEFT_INDEX, RIGHT_INDEX = 19, 20
LEFT_HIP, RIGHT_HIP = 23, 24
LEFT_KNEE, RIGHT_KNEE = 25, 26
LEFT_ANKLE, RIGHT_ANKLE = 27, 28
LEFT_HEEL, RIGHT_HEEL = 29, 30
LEFT_FOOT, RIGHT_FOOT = 31, 32

## Step 1: See the pose

Run the cell below and allow the camera when your browser asks. Step back until your head, shoulders and hips are in view. If you can, get your whole body in, down to your feet: the more of you the model can see, the better it works.

The skeleton is drawn by `canvas.draw_poses()`, using each landmark's `x` and `y`. Points the model isn't sure about (`visibility` under 0.5) are left out.

When the skeleton follows you, click **Stop** (■) and move on to step 2.

In [ ]:
from codetto import graphics2d, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    poses = detect(frame, mirrored=True)
    canvas.draw_poses(poses)
finally:
  camera.stop()

## Step 2: Draw the numbers

This time we'll draw the skeleton ourselves, without the video, using each landmark's `world` position instead of its pixels. A `world` position is three numbers in metres, measured from the middle of your hips:

- **x** goes across the picture. Right is positive.
- **y** goes up and down. Like pixels, it counts **downwards**, so your head has a negative `y` and your feet a positive one.
- **z** goes towards and away from the camera. Negative is closer to the camera.

The camera only sees a flat picture, so how can the model know `z`? It has learned from a huge number of pictures of people what bodies look like and how they bend. From the picture, it makes a very good guess at the 3D shape.

The canvas shows two views of the same landmarks:

- **Front:** `x` across and `y` down, like the camera sees you.
- **Side:** `z` across and `y` down, as if someone were standing to your side. The camera is on the left.

Try pointing at the camera, or reaching one arm forward and the other back. The front view barely changes, but the side view shows exactly what you did.

To turn metres into pixels, `to_canvas()` multiplies by `SCALE` (180 pixels per metre) and adds the middle of each view. Your left side is drawn in blue and your right side in orange.

In [ ]:
import math
from codetto import graphics2d, webcam

WIDTH, HEIGHT = 800, 450
SCALE = 180  # pixels per metre

# Lines to draw between landmarks, for each side of the body.
LEFT_BONES = [(LEFT_SHOULDER, LEFT_ELBOW), (LEFT_ELBOW, LEFT_WRIST), (LEFT_SHOULDER, LEFT_HIP),
              (LEFT_HIP, LEFT_KNEE), (LEFT_KNEE, LEFT_ANKLE), (LEFT_ANKLE, LEFT_FOOT)]
RIGHT_BONES = [(RIGHT_SHOULDER, RIGHT_ELBOW), (RIGHT_ELBOW, RIGHT_WRIST), (RIGHT_SHOULDER, RIGHT_HIP),
               (RIGHT_HIP, RIGHT_KNEE), (RIGHT_KNEE, RIGHT_ANKLE), (RIGHT_ANKLE, RIGHT_FOOT)]
MIDDLE_BONES = [(LEFT_SHOULDER, RIGHT_SHOULDER), (LEFT_HIP, RIGHT_HIP)]

canvas = graphics2d.canvas(WIDTH, HEIGHT)
ctx = canvas.get_context('2d')
camera = webcam.start(mirror=True)

def to_canvas(landmark, view):
  """Turn a landmark's world position (metres) into a point on the canvas."""
  x, y, z = landmark['world']
  if view == 'front':
    return WIDTH / 4 + x * SCALE, HEIGHT / 2 + y * SCALE
  return WIDTH * 3 / 4 + z * SCALE, HEIGHT / 2 + y * SCALE

def visible(landmark):
  return landmark['visibility'] > 0.5

def draw_bones(pose, bones, color, view):
  ctx.stroke_style = color
  ctx.begin_path()
  for start, end in bones:
    if visible(pose[start]) and visible(pose[end]):
      ctx.move_to(*to_canvas(pose[start], view))
      ctx.line_to(*to_canvas(pose[end], view))
  ctx.stroke()

def draw_head(pose, view):
  if visible(pose[LEFT_EAR]) or visible(pose[RIGHT_EAR]):
    left_x, left_y = to_canvas(pose[LEFT_EAR], view)
    right_x, right_y = to_canvas(pose[RIGHT_EAR], view)
    ctx.begin_path()
    ctx.arc((left_x + right_x) / 2, (left_y + right_y) / 2, 0.1 * SCALE, 0, 2 * math.pi)
    ctx.stroke()

def draw_background():
  ctx.fill_style = '#f5f5f5'
  ctx.fill_rect(0, 0, WIDTH, HEIGHT)
  ctx.fill_style = '#444444'
  ctx.font = '18px sans-serif'
  ctx.fill_text('Front', 16, 28)
  ctx.fill_text('Side (camera on the left)', WIDTH / 2 + 16, 28)
  ctx.fill_rect(WIDTH / 2 - 1, 0, 2, HEIGHT)

ctx.line_width = 6
ctx.line_cap = 'round'
try:
  while True:
    frame = camera.capture_array()
    poses = detect(frame, mirrored=True)
    draw_background()
    if poses:
      pose = poses[0]
      for view in ('front', 'side'):
        draw_bones(pose, LEFT_BONES, '#1e88e5', view)
        draw_bones(pose, RIGHT_BONES, '#f4511e', view)
        draw_bones(pose, MIDDLE_BONES, '#555555', view)
        draw_head(pose, view)
finally:
  camera.stop()

## Step 3: Bring it to life

Now for the mannequin. You could put a sphere on every landmark and move it each frame, but it would look like a cloud of floating dots, and it would shake as the model's guesses wobble. A real body has bones that never change length, so we'll build the mannequin the way animators rig a 3D character:

1. **Bones keep their length.** An upper arm is always 28 cm long, so we only need to know which way it **points**. From the landmarks we take the direction from shoulder to elbow, and ignore how far apart the model thinks they are.
2. **Build outwards from the hips.** The hips sit at the middle of the scene. The spine points up from the hips to the chest, each shoulder sits either side of the chest, the elbow is one upper arm from the shoulder, the wrist one forearm from the elbow, and so on down to the hands and feet. In 3D graphics this is called **forward kinematics**.
3. **Smooth the movement.** Each frame, every part turns only part of the way (`FOLLOW`) towards its new direction. This takes out the shake, at the cost of a little delay.
4. **Hold what can't be seen.** If the model can't see a part of you (its `visibility` is low), that part stays where it was. Before the camera has seen it, it hangs straight down. So if only your top half is in view, the mannequin still stands on its legs.
5. **Stand on the floor.** The landmarks are measured from your hips, so at the end we lift the whole body until its lowest foot touches the floor. Squat, and the mannequin squats too.

The parts are **capsules**: cylinders with rounded ends, from `graphics3d.Shapes.Capsule()`. A capsule from one joint to the next, with a ball at each joint, looks like the limb of an artist's wooden mannequin. The chest, hips and head are spheres stretched with `set_scale()`.

The first cell below holds the 3D maths: how to turn a capsule so that it points from one joint to the next. You don't need to read it to use it. The second cell describes the mannequin: the length and thickness of each part, and the `body_joints()` function that works out where every joint goes. Change the numbers to make a different figure!

In [ ]:
#@title 3D maths
import math
import numpy as np

UP = np.array([0.0, 1.0, 0.0])

def unit(arrow):
  """Shrink or stretch an arrow to 1 metre long, keeping its direction."""
  length = np.linalg.norm(arrow)
  return arrow / length if length > 1e-6 else UP

def direction(start, end):
  """The direction from one point to another, as an arrow 1 metre long."""
  return unit(end - start)

def to_scene(landmark):
  """A landmark's world position as a point in the 3D scene. The model
  counts y downwards, like the picture; the 3D scene counts y upwards."""
  x, y, z = landmark['world']
  return np.array([x, -y, z])

def middle(pose, a, b):
  """The point halfway between two landmarks, in the 3D scene."""
  return (to_scene(pose[a]) + to_scene(pose[b])) / 2

def can_see(pose, *landmarks):
  """True if the model is fairly sure all these landmarks are in view."""
  return all(pose[i]['visibility'] > 0.5 for i in landmarks)

def rotation_pointing(d):
  """The rotation, in degrees, that turns an upright shape (like a capsule)
  so that it points along the direction d: tip it over, then turn it around."""
  tip = math.degrees(math.acos(max(-1.0, min(1.0, d[1]))))
  turn = math.degrees(math.atan2(d[0], d[2]))
  return tip, turn, 0

def rotation_facing(across, up):
  """The rotation, in degrees, that turns a shape so that its x axis points
  along `across` and its y axis along `up`. Used for the chest, hips and head."""
  y = unit(up)
  x = across - np.dot(across, y) * y
  if np.linalg.norm(x) < 1e-6:
    x = np.cross(y, [0.0, 0.0, 1.0])
  x = unit(x)
  z = np.cross(x, y)
  tip = math.asin(max(-1.0, min(1.0, -z[1])))
  turn = math.atan2(z[0], z[2])
  roll = math.atan2(x[1], y[1])
  return math.degrees(tip), math.degrees(turn), math.degrees(roll)

def place_between(mesh, start, end):
  """Move and turn a capsule so that it runs from start to end."""
  mesh.set_position(*((start + end) / 2))
  mesh.set_rotation(*rotation_pointing(direction(start, end)))

def place_facing(mesh, position, across, up):
  """Move a shape to position and turn it to face the way the body does."""
  mesh.set_position(*position)
  mesh.set_rotation(*rotation_facing(across, up))

In [ ]:
#@title The mannequin
# Sizes in metres, for a figure about 1.7 m tall.
TORSO = 0.50           # middle of the hips to middle of the shoulders
SHOULDER_WIDTH = 0.38
HIP_WIDTH = 0.22
NECK = 0.24            # middle of the shoulders to middle of the head

# Each limb: the joint at its end, the joint it hangs from, the two landmarks
# that say which way it points, its length and its thickness.
LIMBS = [
  ('left_elbow',  'left_shoulder',  LEFT_SHOULDER,  LEFT_ELBOW,  0.28, 0.09),
  ('left_wrist',  'left_elbow',     LEFT_ELBOW,     LEFT_WRIST,  0.25, 0.075),
  ('left_hand',   'left_wrist',     LEFT_WRIST,     LEFT_INDEX,  0.12, 0.07),
  ('right_elbow', 'right_shoulder', RIGHT_SHOULDER, RIGHT_ELBOW, 0.28, 0.09),
  ('right_wrist', 'right_elbow',    RIGHT_ELBOW,    RIGHT_WRIST, 0.25, 0.075),
  ('right_hand',  'right_wrist',    RIGHT_WRIST,    RIGHT_INDEX, 0.12, 0.07),
  ('left_knee',   'left_hip',       LEFT_HIP,       LEFT_KNEE,   0.42, 0.13),
  ('left_ankle',  'left_knee',      LEFT_KNEE,      LEFT_ANKLE,  0.41, 0.10),
  ('left_toe',    'left_ankle',     LEFT_HEEL,      LEFT_FOOT,   0.17, 0.08),
  ('right_knee',  'right_hip',      RIGHT_HIP,      RIGHT_KNEE,  0.42, 0.13),
  ('right_ankle', 'right_knee',     RIGHT_KNEE,     RIGHT_ANKLE, 0.41, 0.10),
  ('right_toe',   'right_ankle',    RIGHT_HEEL,     RIGHT_FOOT,  0.17, 0.08),
]

FOLLOW = 0.5  # 1 = jump straight to each new pose; smaller is smoother but slower

# Which way each part points before the camera has seen it: standing
# straight, arms down, feet towards the camera. The picture is mirrored,
# so your left side is on the left (negative x).
DOWN = np.array([0.0, -1.0, 0.0])
TO_THE_LEFT = np.array([-1.0, 0.0, 0.0])
directions = {joint: DOWN for joint, *_ in LIMBS}
directions.update({
  'left_toe': np.array([0.0, 0.0, -1.0]), 'right_toe': np.array([0.0, 0.0, -1.0]),
  'spine': UP, 'neck': UP, 'face': np.array([0.0, 0.0, -1.0]),
  'shoulders': TO_THE_LEFT, 'hips': TO_THE_LEFT, 'ears': TO_THE_LEFT,
})

def follow(part, start, end):
  """Turn a part some of the way from where it points now towards the
  direction from start to end."""
  old = directions[part]
  directions[part] = unit(old + (direction(start, end) - old) * FOLLOW)

def turn_towards(pose):
  """Turn every part the model can see towards its new direction."""
  if can_see(pose, LEFT_HIP, RIGHT_HIP, LEFT_SHOULDER, RIGHT_SHOULDER):
    follow('spine', middle(pose, LEFT_HIP, RIGHT_HIP), middle(pose, LEFT_SHOULDER, RIGHT_SHOULDER))
  if can_see(pose, LEFT_SHOULDER, RIGHT_SHOULDER):
    follow('shoulders', to_scene(pose[RIGHT_SHOULDER]), to_scene(pose[LEFT_SHOULDER]))
  if can_see(pose, LEFT_HIP, RIGHT_HIP):
    follow('hips', to_scene(pose[RIGHT_HIP]), to_scene(pose[LEFT_HIP]))
  if can_see(pose, LEFT_EAR, RIGHT_EAR, LEFT_SHOULDER, RIGHT_SHOULDER):
    follow('neck', middle(pose, LEFT_SHOULDER, RIGHT_SHOULDER), middle(pose, LEFT_EAR, RIGHT_EAR))
    follow('ears', to_scene(pose[RIGHT_EAR]), to_scene(pose[LEFT_EAR]))
  if can_see(pose, NOSE, LEFT_EAR, RIGHT_EAR):
    follow('face', middle(pose, LEFT_EAR, RIGHT_EAR), to_scene(pose[NOSE]))
  for joint, parent, start, end, length, thickness in LIMBS:
    if can_see(pose, start, end):
      follow(joint, to_scene(pose[start]), to_scene(pose[end]))

def body_joints(pose=None):
  """Work out where every joint of the mannequin goes for one pose. With
  no pose, every part keeps pointing the way it already does."""
  # 1. Turn the parts the model can see.
  if pose is not None:
    turn_towards(pose)

  # 2. Build the body outwards from the hips, one fixed-length part at a time.
  joints = {'hips': np.zeros(3)}
  joints['chest'] = joints['hips'] + directions['spine'] * TORSO
  joints['left_shoulder'] = joints['chest'] + directions['shoulders'] * SHOULDER_WIDTH / 2
  joints['right_shoulder'] = joints['chest'] - directions['shoulders'] * SHOULDER_WIDTH / 2
  joints['left_hip'] = joints['hips'] + directions['hips'] * HIP_WIDTH / 2
  joints['right_hip'] = joints['hips'] - directions['hips'] * HIP_WIDTH / 2
  joints['head'] = joints['chest'] + directions['neck'] * NECK
  joints['nose'] = joints['head'] + directions['face'] * 0.11
  for joint, parent, start, end, length, thickness in LIMBS:
    joints[joint] = joints[parent] + directions[joint] * length

  # 3. Lift the whole body so that the lowest foot rests on the floor.
  lowest = min(joints[j][1] for j in ('left_ankle', 'right_ankle', 'left_toe', 'right_toe'))
  lift = np.array([0.0, 0.05 - lowest, 0.0])
  return {name: point + lift for name, point in joints.items()}

### Run the mannequin

The cell below builds the scene: a wooden floor, a light, and a mannequin made of capsules (`limbs`), balls (`balls`) and stretched spheres. Then, on every frame, it:

1. takes a picture from the webcam and finds your pose with `detect()`,
2. works out where every joint goes with `body_joints()`,
3. moves the parts with `place_mannequin()`: each capsule between its two joints with `place_between()`, and each ball onto its joint.

Stand back so the camera can see as much of you as possible, and move! Drag the scene to walk around the mannequin and scroll to zoom. Click **Stop** (■) when you're done.

If the mannequin moves slowly, close other tabs and apps: both neural networks run on every frame, so a slower computer may only manage a few updates a second.

In [ ]:
from codetto import graphics3d, webcam

Shapes = graphics3d.Shapes
WOOD = graphics3d.Material.Wood.Oak
JOINTS = graphics3d.Material.Marble.Charcoal

scene = graphics3d.Scene()
scene.set_sky('#dfe7ee')
floor = scene.set_ground(length=4, width=4)
floor.set_material(graphics3d.Material.WoodFloor.PinePlanks)
scene.add_light(1.5, 3, -2)
scene.camera.look_at(0, 0.85, 0)
scene.camera.set_position(0, 1.3, -3.0)

def add_part(mesh, material=WOOD):
  mesh.set_material(material)
  scene.add(mesh)
  return mesh

def add_stretched_sphere(width, height, depth):
  sphere = Shapes.Sphere(diameter=1)
  sphere.set_scale(width, height, depth)
  return add_part(sphere)

# A capsule for each limb. Its length includes the rounded ends, so they are
# centred on the joints at each end.
limbs = {joint: add_part(Shapes.Capsule(diameter=thickness, height=length + thickness))
         for joint, parent, start, end, length, thickness in LIMBS}

# A ball at every joint, a little thicker than the limb that hangs from it.
balls = {}
for joint, parent, start, end, length, thickness in LIMBS:
  if parent not in balls:
    balls[parent] = add_part(Shapes.Sphere(diameter=thickness + 0.03), JOINTS)

shoulder_bar = add_part(Shapes.Capsule(diameter=0.1, height=SHOULDER_WIDTH + 0.1))
waist = add_part(Shapes.Capsule(diameter=0.2, height=0.45))
neck = add_part(Shapes.Capsule(diameter=0.08, height=NECK))
chest = add_stretched_sphere(0.36, 0.34, 0.22)
pelvis = add_stretched_sphere(0.32, 0.20, 0.20)
head = add_stretched_sphere(0.19, 0.24, 0.21)
nose = add_part(Shapes.Sphere(diameter=0.05), JOINTS)

camera = webcam.start(mirror=True)
ctx = scene.get_context('2d')
waiting = None

def show_waiting(now_waiting):
  """Show a message on the scene while nobody is in view."""
  global waiting
  if now_waiting == waiting:
    return
  waiting = now_waiting
  ctx.clear()
  if waiting:
    ctx.fill_style = '#333333'
    ctx.font = 'bold 20px sans-serif'
    ctx.fill_text('Step into view of the camera', 16, 32)

def place_mannequin(joints):
  """Move every part of the mannequin to its joints."""
  for joint, parent, start, end, length, thickness in LIMBS:
    place_between(limbs[joint], joints[parent], joints[joint])
  for joint, ball in balls.items():
    ball.set_position(*joints[joint])

  spine = directions['spine']
  place_between(shoulder_bar, joints['right_shoulder'], joints['left_shoulder'])
  place_between(waist, joints['hips'] + spine * 0.05, joints['hips'] + spine * 0.3)
  place_between(neck, joints['chest'], joints['head'])
  place_facing(chest, joints['hips'] + spine * 0.36, directions['shoulders'], spine)
  place_facing(pelvis, joints['hips'] + spine * 0.03, directions['hips'], spine)
  place_facing(head, joints['head'], directions['ears'], directions['neck'])
  nose.set_position(*joints['nose'])

@scene.on_frame
def move(dt):
  frame = camera.capture_array()
  poses = detect(frame, mirrored=True)
  show_waiting(not poses)
  if poses:
    place_mannequin(body_joints(poses[0]))

place_mannequin(body_joints())  # stand still until someone steps into view
try:
  scene.run()
finally:
  camera.stop()

# Things to Try

- **Your own figure:** change the lengths and thicknesses in `LIMBS`, or swap the wood for another material such as `graphics3d.Material.Chip.CircuitBlue` to build a robot.
- **Smoother or snappier:** try `FOLLOW = 0.2` and `FOLLOW = 1` in the mannequin cell. Which feels better to control?
- **Hands up!** In `move()`, keep what `body_joints()` returns in a variable called `joints`, check whether both wrists are above the head (`joints['left_wrist'][1] > joints['head'][1]`), and change the head's material when they are.
- **Walk around:** the mannequin always stands in the middle, because the `world` landmarks are measured from your hips. Use the hips' `x` in pixels to move the whole mannequin left and right as you walk.
- **Echo:** keep the last 60 sets of joints in a list, and add a second mannequin that copies you two seconds later.

# Check Your Understanding

What would you build if you could control a 3D character with your body? Describe a game, an app or a piece of art.